In [1]:
import gym
import talib
import numpy as np
import pandas as pd
import math
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.utils import get_schedule_fn
from gym import spaces
import torch.nn as nn
import torch as th
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.ppo.policies import MlpPolicy
from stable_baselines3.common.vec_env import SubprocVecEnv
import torch
import os
import torch.nn.functional as F
from stable_baselines3.common.distributions import Distribution
from stable_baselines3.common.policies import MultiInputActorCriticPolicy
from stable_baselines3.common.callbacks import BaseCallback

sequence_len = 1
use_gpu = False

if use_gpu:
    os.environ["PYTORCH_DISABLE_MODEL_LOADING_RAM_OPTIMIZATION"] = "1"
    torch.set_default_dtype(torch.float32)  # Ensure all tensors use float32
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
else:
    device = "cpu"

%run "../helpers/data_manipulator.py"

all_data = None
all_data_np = None

class TradingEnv(gym.Env):
    def __init__(self, data, initial_balance=10000):
        super(TradingEnv, self).__init__()
        self.data = data
        self.tickers = my_stocks
        self.lambda_penalty = 0.002
        self.initial_balance = initial_balance
        self.current_step = 0  # Start at 201 for history
        # self.max_steps = len(self.data[self.data["TICKER"] == my_stocks[0]]) - self.current_step
        self.max_steps = len(self.data) - self.current_step
        self.cash_balance = self.initial_balance
        self.positions = np.zeros(len(my_stocks))
        self.last_prices = np.zeros(len(my_stocks))
        self.total_reward = 1.0
        self.average_buy_prices = np.zeros(len(my_stocks))

        
        num_features = data.shape[3]#  - 2  # Exclude price column
        print(f"num_features: {num_features}")
        
        # Action space: Portfolio allocation percentages (including cash)
        # Using Box with [0,1] range for Dirichlet distribution
        self.action_space = spaces.Box(low=0.0, high=1.0, shape=(len(my_stocks) + 1,), dtype=np.float32)
        
        self.prev_action = np.ones(self.action_space.shape, dtype=np.float32) / (len(my_stocks) + 1)
        
        # Observation space: Last sequence_len days of features
        # self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(len(my_stocks), sequence_len, num_features), dtype=np.float32)

        self.observation_space = spaces.Dict({
            "obs": spaces.Box(low=-np.inf, high=np.inf, shape=(len(my_stocks), sequence_len, num_features), dtype=np.float32),
            "prev_action": spaces.Box(low=0.0, high=1.0, shape=(len(my_stocks) + 1,), dtype=np.float32)
        })


    def reset(self):
        self.current_step = 0
        self.tickers = my_stocks
        self.cash_balance = self.initial_balance
        self.positions = np.zeros(len(my_stocks))
        self.last_prices = np.zeros(len(my_stocks))
        self.total_reward = 1.0
        self.average_buy_prices = np.zeros(len(my_stocks))
        self.prev_action = np.ones(self.action_space.shape, dtype=np.float32) / (len(my_stocks) + 1)
        
        return self._next_observation()

    def _next_observation(self):
        
        return {
            "obs": self.data[self.current_step],  # Current observation
            "prev_action": self.prev_action  # Previously taken action
        }
    
    def step(self, action):
        # No need to clip action here since Dirichlet already ensures all values are in [0,1] and sum to 1
        # But we'll normalize it just to be safe
        epsilon = 1e-8  # Small constant to prevent division by zero
        action /= (np.sum(action) + epsilon)
        
        today_value = self.cash_balance
        tomorrow_value_if_no_change = self.cash_balance + (self.cash_balance * 0.04 / 365)
        tomorrow_value_with_change = 0
        tomorrow_value_with_change_capped = 0
        reward = 0

        for i, ticker in enumerate(my_stocks):
            today_stock_price = all_data_np[i, self.current_step]
            tomorrow_stock_price = all_data_np[i, self.current_step + 1] if self.current_step + 1 < len(all_data_np[i,:]) - 1 else today_stock_price
            today_value += today_stock_price * self.positions[i]
            tomorrow_value_if_no_change += tomorrow_stock_price * self.positions[i]

        # Process all stocks first (excluding cash which is the last element)
        for i, act in enumerate(action[:-1]):  # Exclude the last element (cash)
            ticker = self.tickers[i]
            
            today_stock_price = all_data_np[i, self.current_step]
            tomorrow_stock_price = all_data_np[i, self.current_step + 1] if self.current_step + 1 < len(all_data_np[i,:]) - 1 else today_stock_price
            tomorrow_stock_price_capped = max(min(tomorrow_stock_price, today_stock_price * 1.015), today_stock_price * 0.986)
            
            today_stock_allocation = (today_stock_price * self.positions[i]) / today_value if today_value > 0 else 0
            self.last_prices[i] = today_stock_price
            
            if act > today_stock_allocation and act > 0:
                # Need to buy more of this stock
                desired_invest = act * today_value
                current_invest = today_stock_price * self.positions[i]
                diff = desired_invest - current_invest
                amount_to_buy = math.floor((diff / today_stock_price) * 0.998)  # Account for transaction costs
                if amount_to_buy > 0:
                    cost = (today_stock_price * amount_to_buy) * 1.002
                    if cost <= self.cash_balance:  # Check if we have enough cash
                        self.positions[i] += amount_to_buy
                        self.cash_balance -= cost
                        # Update average buy price
                        if self.positions[i] > 0:
                            self.average_buy_prices[i] = ((self.average_buy_prices[i] * (self.positions[i] - amount_to_buy)) + 
                                                         (today_stock_price * amount_to_buy)) / self.positions[i]
                        else:
                            self.average_buy_prices[i] = today_stock_price
            elif act < today_stock_allocation:
                # Need to sell some of this stock
                desired_invest = act * today_value
                current_invest = today_stock_price * self.positions[i]
                diff = current_invest - desired_invest
                amount_to_sell = math.ceil(diff / today_stock_price)
                amount_to_sell = min(amount_to_sell, self.positions[i])  # Can't sell more than we have
                if amount_to_sell > 0:
                    self.positions[i] -= amount_to_sell
                    self.cash_balance += (today_stock_price * amount_to_sell) * 0.998  # Account for transaction costs
                    
            tomorrow_value_with_change += tomorrow_stock_price * self.positions[i]
            tomorrow_value_with_change_capped += tomorrow_stock_price_capped * self.positions[i]

        # Apply interest to cash balance
        self.cash_balance = self.cash_balance + self.cash_balance * 0.04 / 365
        tomorrow_value_with_change += self.cash_balance
        tomorrow_value_with_change_capped += self.cash_balance

        # Calculate reward as the ratio of tomorrow's portfolio value to the value if no changes were made
        reward += tomorrow_value_with_change_capped / tomorrow_value_if_no_change - 1.0
        
        # Move to next time step
        self.current_step += 1
        done = self.current_step >= self.data.shape[0] - 1
        
        # Store action for next timestep
        self.prev_action = action

        # Log transform the reward
        reward = torch.log(torch.tensor(reward + 1))
        
        obs = self._next_observation()
        self.total_reward *= (1 + reward.item())
        return obs, reward, done, {}


# Dirichlet Distribution implementation
class DirichletDistribution(Distribution):
    def __init__(self, action_dim):
        super(DirichletDistribution, self).__init__()
        self.action_dim = action_dim
        # Remove exploration_param - we'll let PPO's entropy coefficient handle it
        
    def proba_distribution_net(self, latent_dim):
        # Linear layer to produce concentration parameters
        concentration_params = nn.Linear(latent_dim, self.action_dim)
        return concentration_params
        
    def proba_distribution(self, concentration_logits):
        # Simplified - just apply softplus and add a small constant for stability
        self._concentration = F.softplus(concentration_logits) + 0.1
        
        # Create the Dirichlet distribution
        self._dirichlet = torch.distributions.Dirichlet(self._concentration)
        return self
    
    def entropy(self):
        # Simple entropy calculation - no scaling needed
        # SB3's entropy coefficient will handle the importance of entropy
        return self._dirichlet.entropy().unsqueeze(-1)
    
    def actions_from_params(self, concentration_logits, deterministic=False):
        self.proba_distribution(concentration_logits)
        if deterministic:
            actions = self.mode()
        else:
            actions = self.sample()
        log_prob = self.log_prob(actions)
        return actions, log_prob
    
    def log_prob_from_params(self, concentration_logits, actions=None, deterministic=False):
        self.proba_distribution(concentration_logits)
        
        if actions is None:
            if deterministic:
                actions = self.mode()
            else:
                actions = self.sample()
        
        log_prob = self.log_prob(actions)
        return actions, log_prob
        
    def log_prob(self, actions):
        return self._dirichlet.log_prob(actions).unsqueeze(-1)
        
    def sample(self):
        return self._dirichlet.sample()
        
    def mode(self):
        # For Dirichlet, mode is (alpha_i - 1)/(sum(alpha) - K) when alpha_i > 1 for all i
        # If any alpha_i <= 1, the mode puts all mass on the coordinate with largest alpha
        alpha = self._concentration
        alpha_0 = alpha.sum(dim=1, keepdim=True)
        
        # Where alpha > 1, mode is (alpha - 1)/(sum(alpha) - K)
        mode = torch.where(
            alpha > 1.0,
            (alpha - 1.0) / (alpha_0 - self.action_dim),
            torch.zeros_like(alpha)
        )
        
        # If all alpha <= 1, use the highest alpha
        sparse_case = (alpha <= 1.0).all(dim=1, keepdim=True)
        highest_alpha = torch.zeros_like(alpha).scatter_(
            1, 
            alpha.argmax(dim=1, keepdim=True), 
            1.0
        )
        
        mode = torch.where(sparse_case, highest_alpha, mode)
        
        # Normalize to ensure sum to 1
        mode_sum = mode.sum(dim=1, keepdim=True)
        mode = torch.where(
            mode_sum > 0,
            mode / mode_sum,
            torch.ones_like(mode) / self.action_dim
        )
        
        return mode

import torch
import torch.nn.functional as F
from stable_baselines3.common.distributions import Distribution

class ImprovedDirichletDistribution(Distribution):
    def __init__(self, action_dim):
        super().__init__()
        self.action_dim = action_dim
        
    def proba_distribution_net(self, latent_dim):
        # Linear layer to produce concentration parameters with additional regularization
        return torch.nn.Sequential(
            torch.nn.Linear(latent_dim, self.action_dim),
            torch.nn.LayerNorm(self.action_dim),  # Add layer normalization
            torch.nn.LeakyReLU(0.01)  # More flexible activation
        )
        
    def proba_distribution(self, concentration_logits):
        # More dynamic concentration parameter generation
        # Use softplus with learnable temperature and additional regularization
        temperature = torch.nn.Parameter(torch.tensor(1.0))
        self._concentration = F.softplus(concentration_logits * temperature) + 1.0
        
        # Add small random noise to prevent getting stuck
        noise = torch.randn_like(self._concentration) * 0.01
        self._concentration = self._concentration + noise
        
        # Create the Dirichlet distribution
        self._dirichlet = torch.distributions.Dirichlet(self._concentration)
        return self
    
    def entropy(self):
        # More nuanced entropy calculation
        base_entropy = self._dirichlet.entropy()
        
        # Add a regularization term to encourage exploration
        concentration_entropy = -torch.mean(
            torch.log(self._concentration + 1e-8)
        )
        
        # Combine base entropy with concentration entropy
        combined_entropy = base_entropy + 0.1 * concentration_entropy
        
        return combined_entropy.unsqueeze(-1)
    
    def actions_from_params(self, concentration_logits, deterministic=False):
        self.proba_distribution(concentration_logits)
        if deterministic:
            actions = self.mode()
        else:
            actions = self.sample()
        log_prob = self.log_prob(actions)
        return actions, log_prob
    
    def log_prob_from_params(self, concentration_logits, actions=None, deterministic=False):
        self.proba_distribution(concentration_logits)
        
        if actions is None:
            if deterministic:
                actions = self.mode()
            else:
                actions = self.sample()
        
        log_prob = self.log_prob(actions)
        return actions, log_prob
        
    def log_prob(self, actions):
        return self._dirichlet.log_prob(actions).unsqueeze(-1)
        
    def sample(self):
        return self._dirichlet.sample()
        
    def mode(self):
        # Improved mode calculation with additional safeguards
        alpha = self._concentration
        alpha_0 = alpha.sum(dim=1, keepdim=True)
        
        # Where alpha > 1, use standard mode calculation
        mode = torch.where(
            alpha > 1.0,
            (alpha - 1.0) / (alpha_0 - self.action_dim),
            torch.zeros_like(alpha)
        )
        
        # Handle edge cases
        sparse_case = (alpha <= 1.0).all(dim=1, keepdim=True)
        highest_alpha = torch.zeros_like(alpha).scatter_(
            1, 
            alpha.argmax(dim=1, keepdim=True), 
            1.0
        )
        
        mode = torch.where(sparse_case, highest_alpha, mode)
        
        # Normalize and handle potential numerical instabilities
        mode_sum = mode.sum(dim=1, keepdim=True)
        mode = torch.where(
            mode_sum > 0,
            mode / (mode_sum + 1e-8),
            torch.ones_like(mode) / self.action_dim
        )
        
        return mode

import torch
import torch.nn.functional as F
from torch.distributions import Dirichlet

class StandardDirichletDistribution:
    def __init__(self, action_dim):
        self.action_dim = action_dim
        
    def proba_distribution_net(self, latent_dim):
        # Standard network to generate concentration parameters
        return torch.nn.Sequential(
            torch.nn.Linear(latent_dim, self.action_dim),
            torch.nn.Softplus()  # Ensures positive values
        )
        
    def proba_distribution(self, concentration_logits):
        # Convert logits to concentration parameters
        self._concentration = F.softplus(concentration_logits) + 1.0
        
        # Create Dirichlet distribution
        self._dirichlet = Dirichlet(self._concentration)
        return self
    
    def entropy(self):
        # Compute entropy using Dirichlet distribution's method
        return self._dirichlet.entropy().unsqueeze(-1)
    
    def log_prob(self, actions):
        return self._dirichlet.log_prob(actions).unsqueeze(-1)
    
    def sample(self):
        return self._dirichlet.sample()
    
    def get_actions(self, deterministic=False):
        if deterministic:
            # Mode of the distribution
            concentrations = self._concentration
            mode = concentrations / concentrations.sum(dim=-1, keepdim=True)
            return mode
        else:
            # Stochastic sampling
            return self.sample()



class DirichletMultiInputPolicy(MultiInputActorCriticPolicy):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        action_dim = self.action_space.shape[0]
        self.action_dist = StandardDirichletDistribution(action_dim)
    
    def _get_action_dist_from_latent(self, latent_pi):
        # Get the concentration parameters from the action net
        concentration_logits = self.action_net(latent_pi)
        
        # Use our Dirichlet distribution
        return self.action_dist.proba_distribution(concentration_logits)
    
    def evaluate_actions(self, obs, actions):
        # Get features and latent representations
        features = self.extract_features(obs)
        latent_pi, latent_vf = self.mlp_extractor(features)  # Only returns 2 values in recent SB3 versions
        
        # Get distribution from latent
        distribution = self._get_action_dist_from_latent(latent_pi)
        
        # Calculate log probabilities
        log_prob = distribution.log_prob(actions)
        
        # Use our custom entropy calculation
        entropy = distribution.entropy()
        
        # Value function
        values = self.value_net(latent_vf)
        
        return values, log_prob, entropy

all_data = pd.DataFrame()
all_data_val = pd.DataFrame()

for ticker in my_stocks:
    # GET RAW DATA FOR THE TICKER
    stock_file, earnings_file = fetch_finance_data_for_tickers(ticker, "10y", refresh=True, verbose=False, incl_earnings=True)
    data = pd.read_csv(stock_file)
    earnings = pd.read_csv(earnings_file)
    data = data.drop(data[data['Volume'] == 0].index)
    data.reset_index(drop=True, inplace=True)
    data = make_new_features_for(data, earnings=earnings, use_adj_close=True)
    data, data_val, _ = split_data(data,
                                train_cut_date = "2024-07-01",
                                validate_cut_date = "2025-03-01",
                                test_end_date = "2025-03-04",
                                train_start_date = "2017-01-01"
                               )
    normalized_data = normalize_data(data, ticker)
    normalized_data_val = normalize_data(data_val, ticker)
    normalized_data["Close"] = data["Adj Close"]
    normalized_data_val["Close"] = data_val["Adj Close"]
    data = normalized_data
    data_val = normalized_data_val
    all_data = pd.concat([all_data, data], ignore_index=True)
    all_data_val = pd.concat([all_data_val, data_val], ignore_index=True)


features_to_use = lstm_features_v2 + extra_dense_layers_features
    
print("Maximum values:\n", all_data[features_to_use].iloc[0:].max().to_dict())
print("\nMinimum values:\n", all_data[features_to_use].iloc[0:].min().to_dict())
    

obs = []
obs_val = []
for ticker in my_stocks:
    ticker_data = all_data[all_data["TICKER"] == ticker]
    ticker_data_val = all_data_val[all_data_val["TICKER"] == ticker]
    
    dat = ticker_data[features_to_use]
    dat = dat.values  

    dat_val = ticker_data_val[features_to_use]
    dat_val = dat_val.values
    
    # Create the sliding window array
    result = np.array([dat[i - sequence_len+1:i+1] for i in range(sequence_len, len(dat))])
    result_val = np.array([dat_val[i - sequence_len+1:i+1] for i in range(sequence_len, len(dat_val))])
    
    # Result shape: (100 - 20, 21, 3) → (80, 21, 3)
    print(result.shape)  # Should print (80, 21, 3)
    print(result_val.shape)  # Should print (80, 21, 3)

    obs.append(result)
    obs_val.append(result_val)

    

# print(obs.to_numpy())
obs = np.array(obs)
obs = obs.transpose(1, 0, 2, 3)
obs = obs.astype(np.float32)
print(obs.shape)

# print(obs.to_numpy())
obs_val = np.array(obs_val)
obs_val = obs_val.transpose(1, 0, 2, 3)
obs_val = obs_val.astype(np.float32)
print(obs_val.shape)




unique_tickers = all_data["TICKER"].unique()

# Convert to a 2D array where rows = tickers, cols = Close values
all_data_np = np.array([all_data[all_data["TICKER"] == ticker]["Close"].values for ticker in unique_tickers], dtype=object)
all_data_np_val = np.array([all_data_val[all_data_val["TICKER"] == ticker]["Close"].values for ticker in unique_tickers], dtype=object)


print(f"all_data_np.shape: {all_data_np.shape}")

# Setup Environment
env = DummyVecEnv([lambda: TradingEnv(obs)])

env.tickers = my_stocks

def calculate_output_for_cnn(initial_len, cnn_depth=3):
    affect = math.floor((initial_len-1)/2)
    if (cnn_depth == 1): 
        return affect
    else:
        return calculate_output_for_cnn(affect, cnn_depth-1)

class SharedStockNetwork(BaseFeaturesExtractor):
    def __init__(self, observation_space: spaces.Dict, num_stocks, input_dim):

        total_end = 100
        
        obs_dim = observation_space.spaces["obs"].shape[0]
        action_dim = observation_space.spaces["prev_action"].shape[0]
        print(f"obs_dim: {obs_dim}, action_dim: {action_dim}, num_stocks: {num_stocks}")
        total_input_dim = obs_dim + action_dim  # Combined input

        features_dim = num_stocks #+ action_dim # Must match final layer output
        self.daily_dense_output = 1
        cnn_end = 1
        
        super(SharedStockNetwork, self).__init__(observation_space, num_stocks * cnn_end)
        # super(SharedStockNetwork, self).__init__(observation_space, total_end)

        self.shared_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim * sequence_len, 32),
            nn.ReLU(),
        )
        
        self.daily_stock_dense = nn.Sequential(
            nn.Dropout(p=0.18),
            nn.Linear(input_dim, 64),
            nn.LeakyReLU(),
            nn.Dropout(p=0.16),
            nn.Linear(64, 16),
            nn.LeakyReLU(),
            nn.Dropout(p=0.14),
            nn.Linear(16, 16),
            nn.LeakyReLU(),
            nn.Dropout(p=0.12),
            nn.Linear(16, 4),
            nn.LeakyReLU(),
            nn.Dropout(p=0.1),
            nn.Linear(4, self.daily_dense_output),
            nn.Sigmoid(),
        )

        # Create a separate daily_stock_dense for each stock
        self.daily_stock_dense_list = nn.ModuleList([
            nn.Sequential(
                nn.Dropout(p=0.18),
                nn.Linear(input_dim, 16),
                nn.LeakyReLU(),
                nn.Dropout(p=0.16),
                nn.Linear(16, 16),
                nn.LeakyReLU(),
                nn.Dropout(p=0.14),
                nn.Linear(16, 8),
                nn.ReLU(),
                nn.Dropout(p=0.12),
                # nn.Linear(8, 4),
                # nn.ReLU(),
                # nn.Dropout(p=0.1),
                nn.Linear(8, self.daily_dense_output),
                nn.Sigmoid(),
            ) for _ in range(num_stocks)  # Create a unique layer for each stock
        ])

        
        # 1D Convolution across stocks
        self.conv = nn.Sequential(
            # nn.Conv1d(
            #     in_channels=self.daily_dense_output,  # Convolution across stocks, not features
            #     out_channels=8,  # Number of filters
            #     kernel_size=3
            # ),
            # nn.ReLU(),
            # nn.Conv1d(
            #     in_channels=8,  # Convolution across stocks, not features
            #     out_channels=8,  # Number of filters
            #     kernel_size=3,  # Consider 3 neighboring stocks
            #     stride=2
            # ),
            # nn.ReLU(),
            # nn.Conv1d(
            #     in_channels=8,  # Convolution across stocks, not features
            #     out_channels=8,  # Number of filters
            #     kernel_size=3,  # Consider 3 neighboring stocks
            #     stride=2
            # ),
            # nn.ReLU(),
            # nn.Flatten(),
            # nn.Linear(8 * calculate_output_for_cnn(sequence_len-2, 2), 8),
            # nn.ReLU(),
            # nn.Linear(8, cnn_end),
            # nn.ReLU()
            # nn.Sigmoid(),
        )
        # 1D Convolution across stocks
        self.stock_conv = nn.Sequential(
            nn.Conv1d(
                in_channels=cnn_end,  # Convolution across stocks, not features
                out_channels=1000,  # Number of filters
                kernel_size=10
            ),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(1000, total_end),
            nn.ReLU(),
        )

        self.fc_after_combined_cnn = nn.Sequential(
            nn.Linear(num_stocks * cnn_end , num_stocks * cnn_end * 2),
            nn.ReLU(),
            nn.Linear(num_stocks * cnn_end * 2 , num_stocks * cnn_end),
            nn.ReLU(),
        )
        self.prev_action_fc = nn.Sequential(
            nn.Linear(action_dim, action_dim),
            nn.ReLU()
        )
        
        self.fully_connected = nn.Sequential(
            nn.Linear(action_dim + num_stocks, num_stocks),
            nn.ReLU()
        )

        self.flatten = nn.Sequential(
            nn.Flatten()
        )
    
    def forward(self, observations):
        obs = observations["obs"]
        prev_action = observations["prev_action"]

        # print(f"x.shape: {x.shape}")
        batch_size, num_stocks, sequence_len, features_len = obs.shape
        # print(f"batch_size, num_stocks, sequence_len, features_len: {batch_size, num_stocks, sequence_len, features_len}")
        
        # x = x.permute(0, 2, 1)  # Move stock dimension to be iterated
        # shared_outputs = [self.shared_layers(x[:, i, :, :]) for i in range(num_stocks)]
        # combined = torch.cat(shared_outputs, dim=-1)
        # final_layer = self.final_layer(combined)
        # flatten = self.flatten(final_layer)
        # return self.fully_connected(flatten)

        # # Apply a different `daily_stock_dense` layer for each stock
        # daily_results = []
        # for i in range(num_stocks):
        #     daily = obs[:, i, :, :].view(-1, features_len)  # Extract data for stock i
        #     daily_result = self.daily_stock_dense_list[i](daily)  # Use specific layer for stock i
        #     daily_results.append(daily_result)

        # # Stack results back together
        # daily_result = torch.stack(daily_results, dim=1).view(batch_size, num_stocks, sequence_len, self.daily_dense_output)

        daily_results = []
        for i in range(num_stocks):
            # Extract data for stock i across all batches and sequences
            daily = obs[:, i, :, :].reshape(-1, features_len)  # Ensures correct reshaping
            
            # Apply the corresponding dense layer for this stock
            daily_result = self.daily_stock_dense_list[i](daily)  
        
            # Reshape back to match batch and sequence dimensions
            daily_results.append(daily_result.view(batch_size, sequence_len, -1))
        
        # Stack the results to match the original stock dimension
        output = torch.stack(daily_results, dim=1)  # Shape: (batch_size, num_stocks, sequence_len, dense_output_size)


        
        # print(f"obs.shape: {obs.shape}")
        # daily = obs.view(-1, features_len)
        # daily_result = self.daily_stock_dense(daily)
        # print(f"daily_result.shape: {daily_result.shape}")
        
        # Step 3: Reshape back to (batch_size, stocks, sequence)
        # daily_result = daily_result.view(batch_size, num_stocks, sequence_len, self.daily_dense_output)
        # print(f"daily_result.shape2: {daily_result.shape}")

        obs_shaped = output.permute(0, 1, 3, 2)
        if sequence_len > 1:
            stock_outputs = torch.stack([self.conv(obs_shaped[:, i, :, :]) for i in range(num_stocks)], dim=1)
        else:
            stock_outputs = torch.stack([self.flatten(obs_shaped[:, i, :, :]) for i in range(num_stocks)], dim=1)
        
        # stock_outputs = stock_outputs.permute(0, 2, 1)
        # double_cnn_outputs = self.stock_conv(stock_outputs)
        
        # print(f"stock_outputs.shape: {stock_outputs.shape}")
        # all_stocks_output = self.final_layer(stock_outputs)
        
        # print(f"stock_outputs.shape: {stock_outputs.shape}")
        stock_outputs = stock_outputs.flatten(start_dim=1)
        double_cnn_outputs = stock_outputs #self.fc_after_combined_cnn(stock_outputs)
        # print(f"stock_outputs.shape: {stock_outputs.shape}")
        
        ## POLICY
        # stock_outputs = torch.stack([self.shared_layers(obs[:, i, :, :]) for i in range(num_stocks)], dim=1)
        
        
        prev_actions_outputs = self.prev_action_fc(prev_action)
        combined = torch.cat([double_cnn_outputs, prev_actions_outputs], dim=1)
        policy_net = self.fully_connected(combined)

        ## VALUE
        # stock_outputs = torch.stack([self.shared_layers(obs[:, i, :, :]) for i in range(num_stocks)], dim=1)
        # value_net = self.final_layer(stock_outputs)
        
        # combined = torch.cat([all_stocks_output, prev_action], dim=1)
        
        # print(f"ASDASFD:{shared_outputs.shape}")
        # shared_outputs = shared_outputs.permute(0, 2, 1)  # (batch, features_dim, stocks)
        # conv_outputs = self.conv(shared_outputs)
        # x = x.view(batch_size * num_stocks, sequence_len, features_len)  # Flatten stock dimension
        # shared_outputs = self.shared_layers(x)  # Process all stocks in parallel
        # shared_outputs = shared_outputs.view(batch_size, num_stocks, -1)  # Reshape back
        # combined = shared_outputs.flatten(start_dim=1)  # Flatten stock outputs


        # return double_cnn_outputs
        # return all_stocks_output
        return policy_net
        

policy_kwargs = dict(
    features_extractor_class=SharedStockNetwork,
    features_extractor_kwargs={"num_stocks": len(my_stocks), "input_dim": len(features_to_use)},
    net_arch=dict(pi=[32, 16], vf=[32, 16, 8]),  # Deeper networks for Dirichlet
    activation_fn=nn.LeakyReLU
)

# Custom PPO with MAE loss for value function
class PPOWithMAE(PPO):
    def _update_policy(self, loss, approx_kl):
        """Modify loss function to use MAE for value function."""
        # Extract components from PPO loss
        policy_loss, value_loss, entropy_loss = loss

        # Convert MSE to MAE for value function
        value_loss = F.l1_loss(self.rollout_buffer.returns, self.rollout_buffer.values)
        # value_loss = torch.log(torch.cosh(self.rollout_buffer.returns - self.rollout_buffer.values)).mean()


        # Final loss combination (following PPO structure)
        total_loss = policy_loss + self.ent_coef * entropy_loss + self.vf_coef * value_loss

        return super()._update_policy(total_loss, approx_kl)

# Early stopping callback
class StopTrainingCallback(BaseCallback):
    def __init__(self, threshold_entropy=0.0, threshold_policy_grad=-0.1, threshold_explained_var=0.9, verbose=1):
        super().__init__(verbose)
        self.threshold_entropy = threshold_entropy
        self.threshold_policy_grad = threshold_policy_grad
        self.threshold_explained_var = threshold_explained_var

    def _on_step(self) -> bool:
        """ This function is called at each training step. """

        # Access the PPO model's losses from its logger
        try:
            entropy_loss = self.model.logger.name_to_value["train/entropy_loss"]
            policy_gradient_loss = self.model.logger.name_to_value["train/policy_gradient_loss"]
            approx_kl = self.model.logger.name_to_value["train/approx_kl"]
            explained_variance = self.model.logger.name_to_value["train/explained_variance"]
            std = self.model.logger.name_to_value.get("train/std", 0)
        except KeyError:
            if self.verbose:
                print("[WARNING] Could not retrieve some metrics, skipping check.")
            return True  # Continue training if values are not found

        # # 🚨 Stopping conditions
        # if entropy_loss > self.threshold_entropy:
        #     print(f"[STOP] Entropy loss is above {self.threshold_entropy}! Possible bug.")
        #     return False  # Stop training

        # if approx_kl > 0.075:
        #     print(f"[STOP] Approx KL ({approx_kl:.2f}) is too high, stopping training.")
        #     return False  # Stop training
        # if abs(policy_gradient_loss) > self.threshold_policy_grad:
        #     print(f"[STOP] Policy gradient loss is very small ({policy_gradient_loss:.5f}), stopping training.")
        #     return False  # Stop training

        if explained_variance > self.threshold_explained_var and std < 0.3:
            print(f"[STOP] Explained variance ({explained_variance:.2f}) is too high, stopping training.")
            return False  # Stop training

        return True  # Continue training

# Create model with Dirichlet policy
model = PPOWithMAE(
    DirichletMultiInputPolicy,  # Use the custom Dirichlet policy
    env, 
    verbose=1, 
    gamma=0.99,
    batch_size=512,
    learning_rate=0.00025,  # Slightly lower learning rate for Dirichlet
    max_grad_norm=0.5,
    ent_coef=0.001,  # Adjusted for Dirichlet
    clip_range=0.2,
    vf_coef=900.0,
    gae_lambda=0.9,
    policy_kwargs=policy_kwargs,
    device=device
)

# Access the current optimizer
current_optimizer = model.policy.optimizer

# new_optimizer = torch.optim.Adam(model.policy.parameters(), lr=0.001, weight_decay=0.000005)
# model.policy.optimizer = new_optimizer

if use_gpu:
    model.policy.to(torch.float32)  # Convert policy to float32

stop_callback = StopTrainingCallback()
model.learn(total_timesteps=1000000, callback=stop_callback)

# Test Model
done = False
obs = env.reset()

while not done:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, _ = env.step(action)
    
    # Calculate total portfolio value
    total_portfolio_value = env.envs[0].cash_balance + np.sum(env.envs[0].positions * env.envs[0].last_prices)
    cash_percentage = env.envs[0].cash_balance / total_portfolio_value * 100 if total_portfolio_value > 0 else 0
    
    print(f"Step: {env.envs[0].current_step}")
    print(f"Action (allocation): {np.round(action[0], 3)}")
    print(f"Reward: {reward}")
    print(f"Last Prices: {env.envs[0].last_prices}")
    print(f"Positions: {env.envs[0].positions}")
    print(f"Cash: ${env.envs[0].cash_balance:.2f} ({cash_percentage:.1f}%)")
    print(f"Position Value: ${np.sum(env.envs[0].positions * env.envs[0].last_prices):.2f}")
    print(f"Total Portfolio Value: ${total_portfolio_value:.2f}")
    print(f"Cumulative Return: {(env.envs[0].total_reward - 1) * 100:.2f}%")
    print("-" * 50)

fetch_finance_data_for_tickers: about to load from yfinance: MSFT for interval: 10y
DOING EARNINGS
fixed_mean: -0.00011073807603620478, fixed_stddev: 0.011213086281477555
train_data:            Date        Open        High         Low       Close      Volume  \
458  2017-01-03   62.790001   62.840000   62.130001   56.601131  20694100.0   
459  2017-01-04   62.480000   62.750000   62.119999   56.347878  21340000.0   
460  2017-01-05   62.189999   62.660000   62.029999   56.347878  24876000.0   
461  2017-01-06   62.299999   63.150002   62.040001   56.836285  19922900.0   
462  2017-01-09   62.759998   63.080002   62.540001   56.655396  20382700.0   
...         ...         ...         ...         ...         ...         ...   
2338 2024-06-25  448.250000  451.420013  446.750000  448.340454  16747500.0   
2339 2024-06-26  449.000000  453.600006  448.190002  449.543457  16507000.0   
2340 2024-06-27  452.179993  456.170013  451.769989  450.229492  14806300.0   
2341 2024-06-28  453.070007

/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names,

DOING EARNINGS
fixed_mean: -0.0015187177536790813, fixed_stddev: 0.02885841061362865
train_data:            Date        Open        High         Low       Close        Volume  \
458  2017-01-03    2.610000    2.659250    2.484500    2.512894  1.501996e+09   
459  2017-01-04    2.585000    2.637500    2.538250    2.571523  1.199220e+09   
460  2017-01-05    2.613250    2.645500    2.526250    2.506243  9.842960e+08   
461  2017-01-06    2.571250    2.606250    2.530000    2.539745  8.228560e+08   
462  2017-01-09    2.587500    2.700000    2.587500    2.642715  9.162480e+08   
...         ...         ...         ...         ...         ...           ...   
2338 2024-06-25  121.199997  126.500000  119.320000  126.070526  4.257875e+08   
2339 2024-06-26  126.129997  128.119995  122.599998  126.380478  3.629759e+08   
2340 2024-06-27  124.099998  126.410004  122.919998  123.970848  2.525717e+08   
2341 2024-06-28  124.580002  127.709999  122.750000  123.520920  3.155167e+08   
2342 2024-07

/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names,

DOING EARNINGS
fixed_mean: -0.000342506398419089, fixed_stddev: 0.014475688636575036
train_data:            Date        Open        High         Low       Close       Volume  \
458  2017-01-03   37.896000   37.938000   37.384998   37.683498   70422000.0   
459  2017-01-04   37.919498   37.984001   37.709999   37.859001   50210000.0   
460  2017-01-05   38.077499   39.119999   38.013000   39.022499  116602000.0   
461  2017-01-06   39.118000   39.972000   38.924000   39.799500  119724000.0   
462  2017-01-09   39.900002   40.088501   39.588501   39.846001   68922000.0   
...         ...         ...         ...         ...         ...          ...   
2338 2024-06-25  186.809998  188.839996  185.419998  186.339996   45898500.0   
2339 2024-06-26  186.919998  194.800003  186.259995  193.610001   65103900.0   
2340 2024-06-27  195.009995  199.839996  194.199997  197.850006   74397500.0   
2341 2024-06-28  197.729996  198.850006  192.500000  193.250000   76930200.0   
2342 2024-07-01  193.49

...

/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names,

DOING EARNINGS
fixed_mean: 0.0004932118101080091, fixed_stddev: 0.010609372758566118
train_data:            Date        Open        High         Low       Close     Volume  \
458  2017-01-03  104.940002  105.089996  104.209999   82.879761  3741200.0   
459  2017-01-04  105.110001  105.629997  104.790001   83.038185  3029700.0   
460  2017-01-05  104.519997  105.120003  104.190002   82.927292  5087200.0   
461  2017-01-06  104.980003  105.160004  104.120003   82.808510  4109700.0   
462  2017-01-09  104.160004  104.260002  103.250000   81.937347  5603500.0   
...         ...         ...         ...         ...         ...        ...   
2338 2024-06-25  168.080002  168.960007  167.250000  164.683075  4220900.0   
2339 2024-06-26  165.889999  167.009995  164.850006  164.082794  4778300.0   
2340 2024-06-27  166.630005  167.250000  165.270004  163.610428  4326500.0   
2341 2024-06-28  165.300003  166.220001  164.619995  162.301636  8755900.0   
2342 2024-07-01  165.039993  166.059998  162.

/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/opt/homebrew/lib/python3.12/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names,

-----------------------------
| time/              |      |
|    fps             | 1720 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
---------------------------------------
| time/                   |           |
|    fps                  | 1447      |
|    iterations           | 2         |
|    time_elapsed         | 2         |
|    total_timesteps      | 4096      |
| train/                  |           |
|    approx_kl            | 1.6679538 |
|    clip_fraction        | 0.87      |
|    clip_range           | 0.2       |
|    entropy_loss         | 15.8      |
|    explained_variance   | -0.0446   |
|    learning_rate        | 0.00025   |
|    loss                 | 1.03      |
|    n_updates            | 10        |
|    policy_gradient_loss | 0.735     |
|    std                  | 1         |
|    value_loss           | 0.000284  |
---------------------------------------
------------------------------

KeyboardInterrupt: 

In [ ]:
total_pct = 0
for i, ticker in enumerate(my_stocks):
    pct = (all_data_np[i, -1] - all_data_np[i, 0]) / all_data_np[i, 0]
    total_pct +=pct
    print(f"percentage for ticker: {ticker}: {(all_data_np[i, -1] - all_data_np[i, 0]) / all_data_np[i, 0]}")
total_pct /= len(my_stocks)
print(f"percentage if kept equal allocation all the time: {total_pct}")

In [ ]:
## done = np.array([False] * env.num_envs)  # Track done status for all environments

while not np.all(done):  # Continue until all environments are done
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, _ = env.step(action)
    print(f"Step: {env.get_attr("current_step")[0]}, Action: {action}, Reward: {reward}, Position: {env.get_attr("positions")[0]} Total Balance: {env.get_attr("cash_balance")[0] + np.sum(env.get_attr("positions")[0] * env.get_attr("last_prices")[0])} Total Reward: {env.get_attr("total_reward")[0]}, Cash Balance: {env.get_attr("cash_balance")[0]}, Position Balance: {np.sum(env.get_attr("positions")[0] * env.get_attr("last_prices")[0])}")

print(f"\n\n\n Training Done \n\n\n")

In [ ]:
## VALIDATION DATA
total_pct = 0
for i, ticker in enumerate(my_stocks):
    pct = (all_data_np[i, -1] - all_data_np_val[i, 0]) / all_data_np_val[i, 0]
    total_pct +=pct
    print(f"percentage for ticker: {ticker}: {(all_data_np_val[i, -1] - all_data_np_val[i, 0]) / all_data_np_val[i, 0]}")
total_pct /= len(my_stocks)
print(f"percentage if kept equal allocation all the time: {total_pct}")

In [ ]:
# Create environment for new data
env = DummyVecEnv([lambda: TradingEnv(obs_val)])

# Test Model
done = False
obs_val_pre = env.reset()
env.current_step = 0
# last_known_explianed_variance = None    
# cons_counter = 0

while not done:
    action, _states = model.predict(obs_val_pre, deterministic=True)
    obs_val_pre, reward, done, _ = env.step(action)
    print(f"Step: {env.envs[0].current_step}, Action: {action}, Reward: {reward}, Last Price: {env.envs[0].last_prices} Position: {env.envs[0].positions} Total Balance: {env.envs[0].cash_balance + np.sum(env.envs[0].positions * env.envs[0].last_prices)} Total Reward: {env.envs[0].total_reward}, Cash Balance: {env.envs[0].cash_balance}, Position Balance: {np.sum(env.envs[0].positions * env.envs[0].last_prices)}")


In [ ]:
print(len(obs_val))

In [ ]:
to_remove = len(obs_val) - 30
last_30_days = obs_val[to_remove:, :, :, :]

In [ ]:
## VALIDATION DATA
total_pct = 0
for i, ticker in enumerate(my_stocks):
    pct = (all_data_np[i, -1] - all_data_np_val[i, to_remove]) / all_data_np_val[i, to_remove]
    total_pct +=pct
    print(f"percentage for ticker: {ticker}: {(all_data_np_val[i, -1] - all_data_np_val[i, to_remove]) / all_data_np_val[i, to_remove]}")
total_pct /= len(my_stocks)
print(f"percentage if kept equal allocation all the time: {total_pct}")

In [ ]:
# Create environment for new data
env = DummyVecEnv([lambda: TradingEnv(last_30_days)])

# Test Model
done = False
obs_val_pre = env.reset()
env.current_step = 0
# last_known_explianed_variance = None    
# cons_counter = 0

while not done:
    action, _states = model.predict(obs_val_pre, deterministic=True)
    obs_val_pre, reward, done, _ = env.step(action)
    print(f"Step: {env.envs[0].current_step}, Action: {action}, Reward: {reward}, Last Price: {env.envs[0].last_prices} Position: {env.envs[0].positions} Total Balance: {env.envs[0].cash_balance + np.sum(env.envs[0].positions * env.envs[0].last_prices)} Total Reward: {env.envs[0].total_reward}, Cash Balance: {env.envs[0].cash_balance}, Position Balance: {np.sum(env.envs[0].positions * env.envs[0].last_prices)}")


In [ ]:
%run "../helpers/data_manipulator.py"
%run "../helpers/yfinance_data_fetcher.py"
fetch_finance_data_for_tickers("BAC", "5y", refresh=True, verbose=False)

data = yf.download("BAC", period="1mo", actions=True, auto_adjust=False)
print(data)
